# Manual avatar render — admin escape hatch

specs/04-tasks/task-11-talking-head.md: "An owner-side Colab notebook remains as an admin escape hatch for stuck jobs."

Use this when a `render_mode_a` job is stuck (worker crashed mid-render, ZeroGPU Space down, Wav2Lip CPU render taking too long to be worth waiting for). Run SadTalker here on a free Colab GPU instead, then upload the result back via `POST /api/jobs/{id}/import-render` (admin-only — see `backend/app/api/jobs.py`).

**Not automated, not scheduled — a manual, on-demand tool for the owner only.**

## 1. Inputs

You need, for the stuck job:
- The approved portrait (download from `GET /api/avatars/{avatar_id}/portrait` on the VM, or copy from the VM's media directory directly if you have SSH access).
- The rendered TTS audio for the project (same source — the job's stage output, or re-render via `POST /api/projects/{id}/script` -> TTS stage manually if the WAV itself was lost).

Upload both into this Colab session using the file browser on the left, or the cell below.

In [ ]:
from google.colab import files

print('Upload the portrait (PNG/JPG, 1024x1024):')
portrait_upload = files.upload()
portrait_path = next(iter(portrait_upload))

print('Upload the audio (16kHz WAV):')
audio_upload = files.upload()
audio_path = next(iter(audio_upload))


## 2. Vendor SadTalker at the same pinned commit as `hf-space/`

Keep this in sync with whatever commit is pinned in `hf-space/SETUP.md` and `backend/vendor/wav2lip/` (Wav2Lip is the fallback path below if SadTalker fails).

In [ ]:
SADTALKER_COMMIT = 'REPLACE_WITH_THE_PINNED_COMMIT_SHA_FROM_hf-space/SETUP.md'

!git clone https://github.com/OpenTalker/SadTalker.git
%cd SadTalker
!git checkout $SADTALKER_COMMIT
!pip install -r requirements.txt -q
!bash scripts/download_models.sh


## 3. Render

`--still` per the locked Implementation notes (less head wobble, more stable on a single portrait).

In [ ]:
!python inference.py \
  --driven_audio ../{audio_path} \
  --source_image ../{portrait_path} \
  --result_dir ../results \
  --still \
  --preprocess full
%cd ..


## 4. Download the result and import it

Download the resulting MP4 from `results/`, then on the VM (or via the admin UI once one exists — none does yet):

```bash
curl -X POST https://<your-vm>/api/jobs/<job_id>/import-render \
  -F "video=@render.mp4"
```

This requires an admin-role account (see `backend/app/core/deps.py:require_admin` — task-14 will add real login; until then this only works against the dev-mode seeded user with its `role` column manually flipped to `'admin'`).

The endpoint checks the uploaded video's duration against the job's `expected_duration_s` (±2s) as a sanity check — it is not a correctness check on the content.

In [ ]:
import glob
from google.colab import files

rendered = sorted(glob.glob('results/**/*.mp4', recursive=True))
print(rendered)
if rendered:
    files.download(rendered[-1])
